# **Proyecto Final – Optimización e Inteligencia Artificial**
## Predicción del riesgo de incumplimiento crediticio (Home Credit Default Risk): comparación de optimizadores y optimización de hiperparámetros

**Integrantes:**
- Kevin Leandro Ramos Luna
- Daniel Felipe Garzón Acosta
- Ángel Efraín Pimienta Durán
- Daniel Ortega
- Manuel Mera Mera

**Docente:** María C. Torres

**Base de datos:** [Home Credit Default Risk – Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk)

**Estructura del notebook:**

- **Parte 0.** Configuración del entorno
- **Parte 1.** Datos: exploración y preprocesamiento
- **Parte 2.** Modelos de referencia (regresión logística y MLP base)
- **Parte 3.** Comparación de optimizadores para el entrenamiento del MLP
- **Parte 4.** Optimización de hiperparámetros del MLP (Random Search, Bayesiana, Hyperband)
- **Parte 5.** LightGBM + Optuna (TPE vs búsqueda aleatoria)
- **Parte 6.** Evaluación final y comparación global

## PARTE 0: Configuración del entorno

El notebook se ejecuta en **Google Colab con GPU** (*Entorno de ejecución → Cambiar tipo de entorno → GPU*).
Los datos se leen desde Google Drive: subir `application_train.csv` (**descomprimido**) a la carpeta `MyDrive/home-credit-default-risk/`. Si está en otra carpeta de Drive, el notebook lo busca automáticamente.

In [ ]:
!pip install -q keras-tuner optuna lightgbm

In [ ]:
#Importacion de librerias
import os
import sys
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn import metrics

In [ ]:
#TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras

#Semilla para que los resultados sean reproducibles
SEED = 42
keras.utils.set_random_seed(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
#Rutas: en Colab se monta Google Drive, en local se usa la carpeta del proyecto
import glob
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    #Se busca application_train.csv en todo Drive (y en /content si se subio directo a la sesion)
    encontrados = (glob.glob("/content/drive/MyDrive/**/application_train.csv", recursive=True)
                   + glob.glob("/content/application_train.csv"))
    if not encontrados:
        raise FileNotFoundError("No se encontró application_train.csv. Súbalo (descomprimido) a Google Drive, "
                                "idealmente en MyDrive/home-credit-default-risk/, y vuelva a ejecutar esta celda.")
    DATA_DIR = os.path.dirname(encontrados[0])
    FIG_DIR = "/content/drive/MyDrive/OptimizacionEIA_Proyecto/figuras"
else:
    DATA_DIR = "../ProyectoFInal/home-credit-default-risk"
    FIG_DIR = "figuras"
os.makedirs(FIG_DIR, exist_ok=True)
print("Datos en:", DATA_DIR)

def guardar_fig(nombre):
    #Guarda la figura actual en alta resolucion para usarla en el poster
    plt.savefig(f"{FIG_DIR}/{nombre}.png", dpi=300, bbox_inches="tight")

## PARTE 1: Datos – exploración y preprocesamiento

Se usa la tabla principal `application_train.csv`: cada fila es una solicitud de crédito y la variable objetivo **TARGET** indica si el cliente tuvo dificultades de pago (**1**) o no (**0**). Es un problema de **clasificación binaria**.

In [ ]:
#Carga de datos
df = pd.read_csv(f"{DATA_DIR}/application_train.csv")
print("Dimensiones:", df.shape)
print("Variables numéricas:", df.select_dtypes(include="number").shape[1])
print("Variables categóricas:", df.select_dtypes(exclude="number").shape[1])
print("Datos faltantes (total):", df.isna().sum().sum())
df.head()

### Exploración de los datos

In [ ]:
#Distribucion de la variable objetivo
conteo = df["TARGET"].value_counts()
print(conteo)
print("Proporción de incumplimiento: %.2f %%" % (100 * df["TARGET"].mean()))

plt.figure(figsize=(5, 4))
ax = sns.barplot(x=["Paga (0)", "Incumple (1)"], y=conteo.values, hue=["Paga (0)", "Incumple (1)"],
                 palette=["tab:blue", "tab:red"], legend=False)
for i, v in enumerate(conteo.values):
    ax.text(i, v, f"{v:,}\n({100 * v / conteo.sum():.1f} %)", ha="center", va="bottom")
plt.title("Distribución de la variable objetivo")
plt.ylabel("Número de clientes")
plt.ylim(0, conteo.max() * 1.15)
guardar_fig("01_desbalance_target")
plt.show()

El conjunto está **muy desbalanceado** (≈ 8 % de incumplimiento). Un modelo que siempre prediga "paga" tendría ≈ 92 % de *accuracy*, por eso usamos **ROC-AUC** como métrica principal (es también la métrica de la competencia en Kaggle), complementada con PR-AUC, *recall* y F1.

In [ ]:
#Porcentaje de datos faltantes por columna
faltantes = (df.isna().mean() * 100).sort_values(ascending=False)
print("Columnas con faltantes:", (faltantes > 0).sum())
print("Columnas con más del 60 % de faltantes:", (faltantes > 60).sum())
faltantes.head(15)

In [ ]:
#Variables numericas mas correlacionadas con TARGET
corr = df.select_dtypes(include="number").corr()["TARGET"].drop(["TARGET", "SK_ID_CURR"])
top_corr = corr.reindex(corr.abs().sort_values(ascending=False).index).head(10)
print(top_corr)

plt.figure(figsize=(7, 4))
top_corr.sort_values().plot(kind="barh", color=["tab:red" if v > 0 else "tab:blue" for v in top_corr.sort_values()])
plt.title("Top 10 correlaciones con TARGET")
plt.grid(True, axis="x")
guardar_fig("02_correlaciones_target")
plt.show()

In [ ]:
#Distribucion de las variables mas relevantes segun TARGET
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]):
    datos = df[col] / -365 if col == "DAYS_BIRTH" else df[col]
    sns.kdeplot(datos[df["TARGET"] == 0], ax=ax, label="Paga (0)", fill=True)
    sns.kdeplot(datos[df["TARGET"] == 1], ax=ax, label="Incumple (1)", fill=True, color="tab:red")
    ax.set_title("Edad (años)" if col == "DAYS_BIRTH" else col)
    ax.legend()
guardar_fig("03_kde_variables_clave")
plt.show()

### Preprocesamiento

1. **Anomalía en `DAYS_EMPLOYED`:** el valor 365243 (≈ 1000 años) corresponde a pensionados/desempleados → se reemplaza por NaN y se crea la bandera `DAYS_EMPLOYED_ANOM`.
2. **`CODE_GENDER = XNA`:** solo 4 registros, se eliminan.
3. **Ingeniería de variables:** razones financieras (crédito/ingreso, anualidad/ingreso, plazo del crédito, proporción de vida laboral) y combinaciones de las `EXT_SOURCE`.
4. **Columnas con más del 60 % de faltantes** se eliminan.
5. **Codificación:** variables categóricas binarias → 0/1; el resto → *one-hot* (los faltantes quedan como una categoría más).
6. **Partición estratificada** 70 % entrenamiento / 15 % validación / 15 % prueba.
7. **Imputación** (mediana) y **estandarización**, ajustadas **solo con el conjunto de entrenamiento** para no filtrar información.

In [ ]:
datos = df.copy()

#1. Anomalia de DAYS_EMPLOYED
datos["DAYS_EMPLOYED_ANOM"] = (datos["DAYS_EMPLOYED"] == 365243).astype(int)
datos["DAYS_EMPLOYED"] = datos["DAYS_EMPLOYED"].replace(365243, np.nan)

#2. Registros con genero desconocido
datos = datos[datos["CODE_GENDER"] != "XNA"]

#3. Ingenieria de variables
datos["CREDIT_INCOME_RATIO"] = datos["AMT_CREDIT"] / datos["AMT_INCOME_TOTAL"]
datos["ANNUITY_INCOME_RATIO"] = datos["AMT_ANNUITY"] / datos["AMT_INCOME_TOTAL"]
datos["CREDIT_TERM"] = datos["AMT_ANNUITY"] / datos["AMT_CREDIT"]
datos["DAYS_EMPLOYED_RATIO"] = datos["DAYS_EMPLOYED"] / datos["DAYS_BIRTH"]
datos["EXT_SOURCE_MEAN"] = datos[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis=1)
datos["EXT_SOURCE_PROD"] = datos["EXT_SOURCE_1"] * datos["EXT_SOURCE_2"] * datos["EXT_SOURCE_3"]

#4. Columnas con mas del 60 % de faltantes
cols_drop = datos.columns[datos.isna().mean() > 0.60]
datos = datos.drop(columns=cols_drop)
print("Columnas eliminadas por faltantes:", len(cols_drop))
print("Dimensiones:", datos.shape)

In [ ]:
#5. Codificacion de variables categoricas
cat_cols = datos.select_dtypes(exclude="number").columns
binarias = [c for c in cat_cols if datos[c].nunique(dropna=False) == 2]
multiclase = [c for c in cat_cols if c not in binarias]

for col in binarias:
    datos[col] = pd.factorize(datos[col])[0]
datos = pd.get_dummies(datos, columns=multiclase, dummy_na=True)
print("Binarias:", binarias)
print("Categóricas con one-hot:", len(multiclase))

#Caracteristicas y variable objetivo
X = datos.drop(columns=["SK_ID_CURR", "TARGET"]).astype("float32")
y = datos["TARGET"].values.astype("float32")
feature_names = X.columns.tolist()
print("X:", X.shape, " y:", y.shape)

In [ ]:
#6. Particion estratificada: 70 % entrenamiento, 15 % validacion, 15 % prueba
x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
x_valid, x_test, y_valid, y_test = train_test_split(x_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

#7. Imputacion (mediana) y estandarizacion ajustadas solo con entrenamiento
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
x_train = scaler.fit_transform(imputer.fit_transform(x_train)).astype("float32")
x_valid = scaler.transform(imputer.transform(x_valid)).astype("float32")
x_test = scaler.transform(imputer.transform(x_test)).astype("float32")

for nombre, xs, ys in [("entrenamiento", x_train, y_train), ("validación", x_valid, y_valid), ("prueba", x_test, y_test)]:
    print(f"Datos de {nombre}: {xs.shape}  - incumplimiento: {100 * ys.mean():.2f} %")

In [ ]:
#Pesos por clase para compensar el desbalance (se usan en todos los modelos)
pesos = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: float(pesos[0]), 1: float(pesos[1])}
print("Pesos por clase:", class_weight)

### Funciones auxiliares de evaluación

Basadas en las de los talleres, pero adaptadas a **clasificación desbalanceada**:

- El **umbral de decisión** se elige en **validación** (el que maximiza F1) y luego se aplica en prueba.
- La **velocidad de convergencia** se mide como el número de épocas necesarias para alcanzar un AUC de validación ≥ `UMBRAL_AUC`.

In [ ]:
resultados = {}
UMBRAL_AUC = 0.74   #AUC de validacion "aceptable" para medir velocidad de convergencia

def predecir_proba(model, x):
    #Probabilidad de la clase 1 para modelos de sklearn/LightGBM o de Keras
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x)[:, 1]
    return model.predict(x, batch_size=4096, verbose=0).ravel()

def evaluar(nombre, model, history=None, tiempo=np.nan):
    p_valid = predecir_proba(model, x_valid)
    p_test = predecir_proba(model, x_test)
    #Umbral que maximiza F1 en validacion
    prec, rec, umbrales = metrics.precision_recall_curve(y_valid, p_valid)
    f1 = 2 * prec * rec / (prec + rec + 1e-12)
    umbral = umbrales[np.argmax(f1[:-1])]
    y_pred = (p_test >= umbral).astype(int)

    fila = {"Tiempo (s)": round(tiempo, 2)}
    if history is not None:
        val_auc = np.array(history.history["val_auc"])
        alcanzo = np.where(val_auc >= UMBRAL_AUC)[0]
        fila[f"Épocas hasta val AUC≥{UMBRAL_AUC}"] = alcanzo[0] + 1 if len(alcanzo) else np.nan
        fila["Val AUC prom. últimas 5"] = val_auc[-5:].mean()
    fila.update({
        "Val AUC": metrics.roc_auc_score(y_valid, p_valid),
        "Test AUC": metrics.roc_auc_score(y_test, p_test),
        "Test PR-AUC": metrics.average_precision_score(y_test, p_test),
        "Umbral": umbral,
        "Test Recall": metrics.recall_score(y_test, y_pred),
        "Test Precision": metrics.precision_score(y_test, y_pred),
        "Test F1": metrics.f1_score(y_test, y_pred),
    })
    resultados[nombre] = fila
    print(nombre)
    for k in ["Tiempo (s)", "Val AUC", "Test AUC", "Test PR-AUC", "Test F1"]:
        print(f"  {k}: {fila[k]:.4f}")

def graficar(history, titulo):
    #Curvas de aprendizaje: perdida y AUC
    h = pd.DataFrame(history.history)
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    h[["loss", "val_loss"]].plot(ax=ax[0], title="Pérdida (entropía cruzada)")
    h[["auc", "val_auc"]].plot(ax=ax[1], title="ROC-AUC")
    for a in ax:
        a.grid(True)
        a.set_xlabel("Época")
    fig.suptitle(titulo)
    plt.show()

## PARTE 2: Modelos de referencia

Antes de optimizar se construyen dos modelos de referencia:

- **Regresión logística** (`class_weight="balanced"`): modelo lineal clásico en *credit scoring*.
- **MLP base** (Keras): será el "modelo sin optimizar" contra el cual se medirá la mejora de las Partes 3 y 4.

### Regresión logística

In [ ]:
inicio = time.time()
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, C=0.1)
log_reg.fit(x_train, y_train)
tiempo = time.time() - inicio
evaluar("Regresión logística", log_reg, tiempo=tiempo)

### MLP base

Hiperparámetros **fijos** (se mantienen en la comparación de optimizadores de la Parte 3):

- **Capas ocultas:** 3 (256, 128 y 64 neuronas), activación ReLU, *dropout* de 0.3
- **Capa de salida:** 1 neurona con activación sigmoide
- **Función de pérdida:** entropía cruzada binaria
- **Optimizador:** Adam, tasa de aprendizaje 0.001
- **Épocas:** 30 · **Batch size:** 512 (el conjunto de entrenamiento tiene ≈ 215.000 registros)
- **Pesos por clase** para compensar el desbalance

In [ ]:
#Parametros fijos del MLP base
LR = 0.001
EPOCHS = 30
BATCH_SIZE = 512

def construir_mlp(optimizador, capas=(256, 128, 64), dropout=0.3, seed=SEED):
    #Construye y compila el MLP con el optimizador dado
    keras.utils.set_random_seed(seed)
    model = keras.models.Sequential([keras.layers.Input(shape=x_train.shape[1:])])
    for n in capas:
        model.add(keras.layers.Dense(n, activation="relu"))
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(loss="binary_crossentropy",
                  optimizer=optimizador,
                  metrics=[keras.metrics.AUC(name="auc"),
                           keras.metrics.AUC(curve="PR", name="pr_auc")])
    return model

def entrenar(model, epochs=EPOCHS, callbacks=None, verbose=2):
    #Entrena el modelo y devuelve el historial y el tiempo de entrenamiento
    inicio = time.time()
    history = model.fit(x_train, y_train, epochs=epochs, batch_size=BATCH_SIZE,
                        validation_data=(x_valid, y_valid), class_weight=class_weight,
                        callbacks=callbacks, verbose=verbose)
    return history, time.time() - inicio

In [ ]:
model_base = construir_mlp(keras.optimizers.Adam(learning_rate=LR))
model_base.summary()
history_base, tiempo = entrenar(model_base)
evaluar("MLP base (Adam)", model_base, history_base, tiempo)
graficar(history_base, "Curva de aprendizaje - MLP base")

In [ ]:
tabla_base = pd.DataFrame(resultados).T
tabla_base

### Conclusiones Parte 2

- _TODO: comparar el AUC de la regresión logística vs. el MLP base._
- _TODO: ¿hay sobreajuste en las curvas del MLP? (brecha entre `auc` y `val_auc`)._

## PARTE 3: Comparación de optimizadores para el entrenamiento del MLP

Misma arquitectura y mismos hiperparámetros que el MLP base (tasa de aprendizaje 0.001, 30 épocas, batch 512). Solo cambia el optimizador:

| Vistos en clase | De la literatura |
|---|---|
| SGD, Momentum, Nesterov, RMSprop, Adam | Adagrad, Nadam, AdamW |

Cada optimizador se entrena con **3 semillas** y se reporta la media ± la desviación estándar.

**Métricas:** AUC de validación por época, épocas hasta AUC ≥ `UMBRAL_AUC`, tiempo de entrenamiento y AUC de prueba.

In [ ]:
#Funcion para crear una instancia nueva de cada optimizador
def crear_optimizador(nombre, lr=LR):
    return {"SGD":      lambda: keras.optimizers.SGD(learning_rate=lr),
            "Momentum": lambda: keras.optimizers.SGD(learning_rate=lr, momentum=0.9),
            "Nesterov": lambda: keras.optimizers.SGD(learning_rate=lr, momentum=0.9, nesterov=True),
            "Adagrad":  lambda: keras.optimizers.Adagrad(learning_rate=lr),
            "RMSprop":  lambda: keras.optimizers.RMSprop(learning_rate=lr),
            "Adam":     lambda: keras.optimizers.Adam(learning_rate=lr),
            "Nadam":    lambda: keras.optimizers.Nadam(learning_rate=lr),
            "AdamW":    lambda: keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-4)}[nombre]()

OPTIMIZADORES = ["SGD", "Momentum", "Nesterov", "Adagrad", "RMSprop", "Adam", "Nadam", "AdamW"]
SEMILLAS = [42, 7, 123]

### Entrenamiento de los 24 modelos (8 optimizadores × 3 semillas)

Cada entrenamiento se guarda en un archivo JSON en Drive apenas termina. Si Colab se desconecta, al volver a ejecutar la celda **solo se entrenan los modelos que faltan**.

In [ ]:
#Archivo donde se guardan las corridas (metricas + historial por epoca)
RES_DIR = os.path.join(os.path.dirname(FIG_DIR), "resultados")
os.makedirs(RES_DIR, exist_ok=True)
ARCHIVO_OPT = f"{RES_DIR}/parte3_optimizadores.json"
corridas = {}
if os.path.exists(ARCHIVO_OPT):
    with open(ARCHIVO_OPT) as f:
        corridas = json.load(f)
print("Corridas ya guardadas:", len(corridas), "de", len(OPTIMIZADORES) * len(SEMILLAS))

#Se guardan los resultados de la parte 2 y se reinicia el diccionario
resultados_base = dict(resultados)
resultados = {}

In [ ]:
for nombre in OPTIMIZADORES:
    for semilla in SEMILLAS:
        clave = f"{nombre} (seed {semilla})"
        if clave in corridas:
            print(clave, "-> ya entrenado, se omite")
            continue
        keras.backend.clear_session()
        model = construir_mlp(crear_optimizador(nombre), seed=semilla)
        history, tiempo = entrenar(model, verbose=0)
        evaluar(clave, model, history, tiempo)
        corridas[clave] = {"Optimizador": nombre, "Semilla": semilla,
                           **{k: float(v) for k, v in resultados[clave].items()},
                           "historia": {k: [float(x) for x in v] for k, v in history.history.items()}}
        with open(ARCHIVO_OPT, "w") as f:
            json.dump(corridas, f)

### Comparación de optimizadores

In [ ]:
#Tabla por corrida y resumen por optimizador (media y desviacion de las 3 semillas)
COL_EPOCAS = f"Épocas hasta val AUC≥{UMBRAL_AUC}"
COLS = ["Tiempo (s)", COL_EPOCAS, "Val AUC prom. últimas 5", "Test AUC", "Test PR-AUC", "Test F1"]

tabla_corridas = pd.DataFrame([{k: v for k, v in c.items() if k != "historia"} for c in corridas.values()])
resumen_opt = tabla_corridas.groupby("Optimizador")[COLS].agg(["mean", "std"]).reindex(OPTIMIZADORES)
alcanzan = tabla_corridas.groupby("Optimizador")[COL_EPOCAS].count().reindex(OPTIMIZADORES)

#Tabla legible "media ± desviacion"
formatos = {"Tiempo (s)": "{:.1f}", COL_EPOCAS: "{:.1f}"}
tabla_opt = pd.DataFrame({
    c: resumen_opt[c]["mean"].map(formatos.get(c, "{:.4f}").format) + " ± "
       + resumen_opt[c]["std"].map(formatos.get(c, "{:.4f}").format)
    for c in COLS})
tabla_opt[COL_EPOCAS] = tabla_opt[COL_EPOCAS].where(alcanzan > 0, "no alcanza")
tabla_opt[f"Semillas que alcanzan {UMBRAL_AUC}"] = alcanzan.astype(str) + f"/{len(SEMILLAS)}"
tabla_opt.sort_values("Val AUC prom. últimas 5", ascending=False)

In [ ]:
#Colores fijos por optimizador (se usan en todas las figuras y en el poster)
COLORES = dict(zip(OPTIMIZADORES, plt.cm.tab10.colors))
VISTOS_EN_CLASE = ["SGD", "Momentum", "Nesterov", "RMSprop", "Adam"]

def curva_media(nombre, metrica):
    #Media y desviacion por epoca de una metrica entre las semillas de un optimizador
    curvas = np.array([c["historia"][metrica] for c in corridas.values() if c["Optimizador"] == nombre])
    return curvas.mean(axis=0), curvas.std(axis=0)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for nombre in OPTIMIZADORES:
    estilo = "-" if nombre in VISTOS_EN_CLASE else "--"
    for a, metrica in zip(ax, ["val_auc", "val_loss"]):
        media, std = curva_media(nombre, metrica)
        epocas = np.arange(1, len(media) + 1)
        a.plot(epocas, media, estilo, color=COLORES[nombre], label=nombre, linewidth=2)
        a.fill_between(epocas, media - std, media + std, color=COLORES[nombre], alpha=0.15)
ax[0].axhline(UMBRAL_AUC, color="gray", linestyle=":", label=f"Umbral AUC = {UMBRAL_AUC}")
ax[0].set_title("ROC-AUC de validación por época")
ax[1].set_title("Pérdida de validación (entropía cruzada) por época")
for a in ax:
    a.set_xlabel("Época")
    a.grid(True)
ax[0].legend(ncol=2)
plt.suptitle("Comparación de optimizadores (media ± desviación de 3 semillas) — línea continua: vistos en clase, discontinua: literatura")
guardar_fig("04_curvas_optimizadores")
plt.show()

In [ ]:
#Barras: AUC de prueba, tiempo y velocidad de convergencia
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
colores = [COLORES[n] for n in OPTIMIZADORES]
graficas = [("Test AUC", "ROC-AUC de prueba"),
            ("Tiempo (s)", "Tiempo de entrenamiento (s)"),
            (COL_EPOCAS, f"Épocas hasta val AUC ≥ {UMBRAL_AUC}")]
for a, (col, titulo) in zip(ax, graficas):
    posiciones = np.arange(len(OPTIMIZADORES))
    a.bar(posiciones, resumen_opt[col]["mean"].fillna(0), yerr=resumen_opt[col]["std"].fillna(0),
          color=colores, capsize=4)
    a.set_xticks(posiciones, OPTIMIZADORES, rotation=45)
    a.set_title(titulo)
    a.grid(True, axis="y")
#Zoom en el eje del AUC para que se noten las diferencias
auc_min = resumen_opt["Test AUC"]["mean"].min()
ax[0].set_ylim(auc_min - 0.02, resumen_opt["Test AUC"]["mean"].max() + 0.01)
#Marcar los optimizadores que no alcanzan el umbral
for i, n in enumerate(OPTIMIZADORES):
    if alcanzan[n] == 0:
        ax[2].text(i, 0, "no\nalcanza", ha="center", va="bottom", color=COLORES[n], fontweight="bold")
guardar_fig("05_barras_optimizadores")
plt.show()

In [ ]:
#Seleccion del optimizador: mayor AUC de validacion (promedio de las ultimas 5 epocas y de las 3 semillas).
#Se usa VALIDACION, no prueba, para no sesgar la evaluacion final.
ranking = resumen_opt["Val AUC prom. últimas 5"]["mean"].sort_values(ascending=False)
mejor_opt = ranking.index[0]
top2_opt = ranking.index[:2].tolist()   #se usan como opciones en la busqueda de la Parte 4
print("Ranking (val AUC):")
print(ranking.round(4))
print("\nOptimizador seleccionado:", mejor_opt)
print("Dos mejores (para la Parte 4):", top2_opt)

#Velocidad de convergencia relativa a SGD (evidencia de la necesidad de optimizadores avanzados)
ep_mejor = resumen_opt[COL_EPOCAS]["mean"][mejor_opt]
if alcanzan["SGD"] == 0:
    print(f"\nSGD no alcanza AUC {UMBRAL_AUC} en {EPOCHS} épocas; {mejor_opt} lo alcanza en {ep_mejor:.1f} épocas.")
else:
    print(f"\n{mejor_opt} alcanza AUC {UMBRAL_AUC} {resumen_opt[COL_EPOCAS]['mean']['SGD'] / ep_mejor:.1f} veces más rápido que SGD.")

### Conclusiones Parte 3

_Completar con los números de la tabla y las figuras. Puntos a cubrir:_

- **Velocidad de convergencia:** ¿cuántas épocas necesita SGD para llegar a AUC ≥ 0.74 frente a Momentum/Nesterov y los adaptativos? ¿Alguno no lo alcanza?
- **Desempeño final:** ¿qué optimizador logra el mayor AUC de validación y de prueba? ¿Las diferencias entre los adaptativos (RMSprop, Adam, Nadam, AdamW) son mayores que la desviación entre semillas?
- **Costo:** el tiempo por época es casi igual para todos (mismo número de actualizaciones); la diferencia real está en cuántas épocas necesitan.
- **Estabilidad:** ¿qué optimizador tiene menor desviación entre semillas? ¿Alguno muestra sobreajuste (la pérdida de validación sube al final)?
- **Selección:** optimizador elegido y los dos mejores que pasan a la búsqueda de hiperparámetros de la Parte 4.

## PARTE 4: Optimización de hiperparámetros del MLP (Keras Tuner)

**Espacio de búsqueda:**

| Hiperparámetro | Valores |
|---|---|
| Número de capas ocultas | 1 – 4 |
| Neuronas por capa | 32 – 512 (paso 32) |
| *Dropout* | 0.0 – 0.5 |
| Regularización L2 | {0, 1e-5, 1e-4, 1e-3} |
| Tasa de aprendizaje | 1e-4 – 1e-2 (escala logarítmica) |
| Optimizador | los 2 mejores de la Parte 3 |

**Estrategias** (mismo presupuesto, objetivo = maximizar `val_auc`):

1. Random Search
2. Optimización Bayesiana
3. Hyperband

In [ ]:
import keras_tuner as kt

#TODO: def build_model(hp): ... (usar kt.Objective("val_auc", direction="max"))
#TODO: tuner_rs = kt.RandomSearch(...), tuner_bo = kt.BayesianOptimization(...), tuner_hb = kt.Hyperband(...)
#      directory en Drive y overwrite=False para poder reanudar si Colab se desconecta
#TODO: callback EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True)
#TODO: grafica "mejor val_auc acumulado vs numero de prueba" para las 3 estrategias -> guardar_fig("06_hpo_convergencia")
#TODO: numero de combinaciones que tendria un Grid Search sobre este espacio (argumento de la necesidad de optimizacion)
#TODO: reentrenar la mejor configuracion y evaluar("MLP optimizado (...)", ...)

### Conclusiones Parte 4

- _TODO_

## PARTE 5: LightGBM + Optuna

*Gradient boosting* (LightGBM) es el estado del arte en datos tabulares. Se compara su versión por defecto contra versiones optimizadas con **Optuna**, usando dos muestreadores con el mismo número de pruebas:

- **TPE** (*Tree-structured Parzen Estimator*, optimización bayesiana)
- **RandomSampler** (búsqueda aleatoria)

In [ ]:
import lightgbm as lgb
import optuna

#TODO: LightGBM por defecto -> evaluar("LightGBM (por defecto)", ...)
#TODO: def objective(trial): num_leaves, learning_rate, max_depth, min_child_samples,
#      subsample, colsample_bytree, reg_alpha, reg_lambda -> AUC de validacion (con early stopping)
#TODO: estudio con optuna.samplers.TPESampler(seed=SEED) y con optuna.samplers.RandomSampler(seed=SEED)
#TODO: grafica del mejor AUC acumulado vs prueba (TPE vs Random) -> guardar_fig("07_optuna_convergencia")
#TODO: importancia de hiperparametros y de variables

### Conclusiones Parte 5

- _TODO_

## PARTE 6: Evaluación final y comparación global

In [ ]:
#TODO: tabla resumen de todos los modelos (Val AUC, Test AUC, PR-AUC, Recall, F1, Tiempo)
#TODO: curvas ROC de los modelos finales en una sola figura -> guardar_fig("08_curvas_roc")
#TODO: matriz de confusion del mejor modelo en prueba (con el umbral elegido en validacion) -> guardar_fig("09_matriz_confusion")

## Conclusiones generales

- _TODO_

## Referencias

- Home Credit Group. (2018). *Home Credit Default Risk* [Competencia de Kaggle]. https://www.kaggle.com/competitions/home-credit-default-risk
- Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (2.ª ed.). O'Reilly.
- Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR*.
- Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR*.
- Dozat, T. (2016). Incorporating Nesterov momentum into Adam. *ICLR Workshop*.
- Bergstra, J., & Bengio, Y. (2012). Random search for hyper-parameter optimization. *JMLR, 13*, 281–305.
- Snoek, J., Larochelle, H., & Adams, R. P. (2012). Practical Bayesian optimization of machine learning algorithms. *NeurIPS*.
- Li, L., Jamieson, K., DeSalvo, G., Rostamizadeh, A., & Talwalkar, A. (2018). Hyperband: A novel bandit-based approach to hyperparameter optimization. *JMLR, 18*(185), 1–52.
- Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A next-generation hyperparameter optimization framework. *KDD*.
- Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *NeurIPS*.